In [1]:
# Week 3 Day 1 — Tensors
# Goal: Understand tensors using CT imaging context

import torch
import numpy as np

# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")
print("✓ Setup complete")

PyTorch version: 2.11.0+cu128
Device: cuda
✓ Setup complete


In [2]:
# Cell 2 — Tensor dimensions
# Connecting to medical physics data you know

# 0D tensor — single number
single_hu = torch.tensor(42)
print(f"0D tensor: {single_hu}")
print(f"   Shape: {single_hu.shape}")
print(f"   Dims:  {single_hu.ndim}")

# 1D tensor — list of numbers
dose_curve = torch.tensor([10.0, 45.0, 80.0, 95.0, 100.0])
print(f"\n1D tensor: {dose_curve}")
print(f"   Shape: {dose_curve.shape}")
print(f"   Dims:  {dose_curve.ndim}")

# 2D tensor — CT slice (small example)
ct_slice = torch.tensor([[100, 200, 150],
                         [300, 250, 180],
                         [-500,  40,  60]])
print(f"\n2D tensor (CT patch):")
print(ct_slice)
print(f"   Shape: {ct_slice.shape}")
print(f"   Dims:  {ct_slice.ndim}")

# 3D tensor — CT volume
ct_volume = torch.zeros(300, 512, 512)
print(f"\n3D tensor (CT volume) shape: {ct_volume.shape}")
print(f"   Dims: {ct_volume.ndim}")

# 4D tensor — batch for neural network
ct_batch = torch.zeros(8, 1, 512, 512)
print(f"\n4D tensor (batch) shape: {ct_batch.shape}")
print(f"   Dims: {ct_batch.ndim}")

0D tensor: 42
   Shape: torch.Size([])
   Dims:  0

1D tensor: tensor([ 10.,  45.,  80.,  95., 100.])
   Shape: torch.Size([5])
   Dims:  1

2D tensor (CT patch):
tensor([[ 100,  200,  150],
        [ 300,  250,  180],
        [-500,   40,   60]])
   Shape: torch.Size([3, 3])
   Dims:  2

3D tensor (CT volume) shape: torch.Size([300, 512, 512])
   Dims: 3

4D tensor (batch) shape: torch.Size([8, 1, 512, 512])
   Dims: 4


In [3]:
# Cell 3 — Tensor operations
# Operations on CT data

# Create two small CT patches
patch_a = torch.tensor([100.0, 200.0, 300.0])
patch_b = torch.tensor([10.0,  20.0,  30.0])

# Basic operations
print(f"patch_a:        {patch_a}")
print(f"patch_b:        {patch_b}")
print(f"Addition:       {patch_a + patch_b}")
print(f"Subtraction:    {patch_a - patch_b}")
print(f"Multiplication: {patch_a * patch_b}")
print(f"Division:       {patch_a / patch_b}")

# Statistics — useful for HU analysis
print(f"\nMean HU:    {patch_a.mean():.1f}")
print(f"Max HU:     {patch_a.max():.1f}")
print(f"Min HU:     {patch_a.min():.1f}")
print(f"Std HU:     {patch_a.std():.1f}")

# Normalisation — HU to 0-1 range
hu_min = patch_a.min()
hu_max = patch_a.max()
normalised = (patch_a - hu_min) / (hu_max - hu_min)
print(f"\nOriginal:   {patch_a}")
print(f"Normalised: {normalised}")

patch_a:        tensor([100., 200., 300.])
patch_b:        tensor([10., 20., 30.])
Addition:       tensor([110., 220., 330.])
Subtraction:    tensor([ 90., 180., 270.])
Multiplication: tensor([1000., 4000., 9000.])
Division:       tensor([10., 10., 10.])

Mean HU:    200.0
Max HU:     300.0
Min HU:     100.0
Std HU:     100.0

Original:   tensor([100., 200., 300.])
Normalised: tensor([0.0000, 0.5000, 1.0000])


In [4]:
# Cell 4 — dtype conversion
# Critical rule: images → float32, labels → int64

# CT image starts as integer
ct_image = torch.tensor([100, 200, -500, 0, 400])
print(f"CT image dtype:    {ct_image.dtype}")

# Convert to float32 for neural network
ct_image = ct_image.float()
print(f"After .float():    {ct_image.dtype}")
print(f"Values:            {ct_image}")

# Segmentation mask stays as integer
mask = torch.tensor([0, 1, 0, 0, 1])
print(f"\nMask dtype:        {mask.dtype}")
print(f"Mask values:       {mask}")

# What happens if you forget to convert
# and try to use integer in network
try:
    # Neural networks need float
    # this will cause an error
    result = ct_image + mask.float()
    print(f"\nCombined result:   {result}")
    print(f"Combined dtype:    {result.dtype}")
except Exception as e:
    print(f"\nError: {e}")

print("\n✓ Rule: images=float32, labels=int64")

CT image dtype:    torch.int64
After .float():    torch.float32
Values:            tensor([ 100.,  200., -500.,    0.,  400.])

Mask dtype:        torch.int64
Mask values:       tensor([0, 1, 0, 0, 1])

Combined result:   tensor([ 100.,  201., -500.,    0.,  401.])
Combined dtype:    torch.float32

✓ Rule: images=float32, labels=int64


In [6]:
# Cell 5 — NumPy to PyTorch conversion
# CT data loads as NumPy → must convert to tensor

import numpy as np

# Simulate CT slice loaded with pydicom
ct_numpy = np.random.randint(-1000, 400,
                              size=(512, 512),
                              dtype=np.int16)

print("── NumPy array ──")
print(f"Shape:  {ct_numpy.shape}")
print(f"dtype:  {ct_numpy.dtype}")
print(f"Min HU: {ct_numpy.min()}")
print(f"Max HU: {ct_numpy.max()}")

# Convert to float32 first
ct_float = ct_numpy.astype(np.float32)

# Convert NumPy → PyTorch
ct_tensor = torch.from_numpy(ct_float)

print("\n── PyTorch tensor ──")
print(f"Shape:  {ct_tensor.shape}")
print(f"dtype:  {ct_tensor.dtype}")
print(f"Min HU: {ct_tensor.min():.1f}")
print(f"Max HU: {ct_tensor.max():.1f}")

# Convert back to NumPy
ct_back = ct_tensor.numpy()
print("\n── Back to NumPy ──")
print(f"Shape: {ct_back.shape}")
print(f"dtype: {ct_back.dtype}")

── NumPy array ──
Shape:  (512, 512)
dtype:  int16
Min HU: -1000
Max HU: 399

── PyTorch tensor ──
Shape:  torch.Size([512, 512])
dtype:  torch.float32
Min HU: -1000.0
Max HU: 399.0

── Back to NumPy ──
Shape: (512, 512)
dtype: float32


In [7]:
# Cell 6 — Complete CT pipeline
# Raw CT → network ready tensor

print("── Step 1: Simulate CT slice ──")
ct_numpy = np.random.randint(-1000, 400,
                              size=(512, 512),
                              dtype=np.int16)
print(f"Shape: {ct_numpy.shape}  dtype: {ct_numpy.dtype}")

print("\n── Step 2: Convert to float32 ──")
ct_tensor = torch.from_numpy(ct_numpy.astype(np.float32))
print(f"Shape: {ct_tensor.shape}  dtype: {ct_tensor.dtype}")

print("\n── Step 3: Move to GPU ──")
ct_tensor = ct_tensor.to(device)
print(f"Shape: {ct_tensor.shape}  device: {ct_tensor.device}")

print("\n── Step 4: Normalise HU values to 0-1 ──")
hu_min = -1000.0
hu_max =  400.0
ct_tensor = (ct_tensor - hu_min) / (hu_max - hu_min)
print(f"Min value: {ct_tensor.min():.3f}")
print(f"Max value: {ct_tensor.max():.3f}")

print("\n── Step 5: Add channel dimension ──")
ct_tensor = ct_tensor.unsqueeze(0)
print(f"Shape: {ct_tensor.shape}")

print("\n── Step 6: Add batch dimension ──")
ct_tensor = ct_tensor.unsqueeze(0)
print(f"Shape: {ct_tensor.shape}")

print("\n── Final result ──")
print(f"Shape:  {ct_tensor.shape}")
print(f"dtype:  {ct_tensor.dtype}")
print(f"device: {ct_tensor.device}")
print(f"Range:  {ct_tensor.min():.3f} to {ct_tensor.max():.3f}")
print("\n✓ CT slice ready for neural network")

── Step 1: Simulate CT slice ──
Shape: (512, 512)  dtype: int16

── Step 2: Convert to float32 ──
Shape: torch.Size([512, 512])  dtype: torch.float32

── Step 3: Move to GPU ──
Shape: torch.Size([512, 512])  device: cuda:0

── Step 4: Normalise HU values to 0-1 ──
Min value: 0.000
Max value: 0.999

── Step 5: Add channel dimension ──
Shape: torch.Size([1, 512, 512])

── Step 6: Add batch dimension ──
Shape: torch.Size([1, 1, 512, 512])

── Final result ──
Shape:  torch.Size([1, 1, 512, 512])
dtype:  torch.float32
device: cuda:0
Range:  0.000 to 0.999

✓ CT slice ready for neural network
